<a href="https://colab.research.google.com/github/alurusaivahini-dotcom/Trend-pulse/blob/main/task2_data_processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import json
import os
import time
from datetime import datetime
TOP_STORIES_URL = "https://hacker-news.firebaseio.com/v0/topstories.json"
ITEM_URL = "https://hacker-news.firebaseio.com/v0/item/{}.json"
headers = {"User-Agent": "TrendPulse/1.0"}
CATEGORIES = {
    "technology": [
        "ai", "software", "tech", "code", "computer",
        "data", "cloud", "api", "gpu", "llm"
    ],
    "worldnews": [
        "war", "government", "country", "president",
        "election", "climate", "attack", "global"
    ],
    "sports": [
        "nfl", "nba", "fifa", "sport", "game",
        "team", "player", "league", "championship"
    ],
    "science": [
        "research", "study", "space", "physics",
        "biology", "discovery", "nasa", "genome"
    ],
    "entertainment": [
        "movie", "film", "music", "netflix",
        "game", "book", "show", "award", "streaming"
    ]
}
try:
    response = requests.get(TOP_STORIES_URL, headers=headers)
    response.raise_for_status()
    top_story_ids = response.json()[:500]
except Exception as e:
    print("Failed to fetch top stories:", e)
    top_story_ids = []
all_stories = []

for category, keywords in CATEGORIES.items():
    print(f"\nCollecting stories for category: {category}")
    category_count = 0

    for story_id in top_story_ids:
        if category_count >= 25:
            break
        try:
            story_response = requests.get(
                ITEM_URL.format(story_id),
                headers=headers
            )

            if story_response.status_code != 200:
                print(f"Failed to fetch story {story_id}")
                continue
            story = story_response.json()
            if not story or "title" not in story:
                continue
            title_lower = story["title"].lower()
            if any(keyword in title_lower for keyword in keywords):
                story_data = {
                    "post_id": story.get("id"),
                    "title": story.get("title"),
                    "category": category,
                    "score": story.get("score", 0),
                    "num_comments": story.get("descendants", 0),
                    "author": story.get("by", "unknown"),
                    "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                }

                all_stories.append(story_data)
                category_count += 1

        except Exception as e:
            print(f"Error fetching story {story_id}: {e}")
            continue
    time.sleep(2)
os.makedirs("data", exist_ok=True)

date_str = datetime.now().strftime("%Y%m%d")
file_path = f"data/trends_{date_str}.json"

with open(file_path, "w", encoding="utf-8") as f:
    json.dump(all_stories, f, indent=4)

print(f"\nCollected {len(all_stories)} stories. Saved to {file_path}")







Collected 110 stories. Saved to data/trends_20260411.json


In [ ]:
# Question 2
# LOad the JSON FILES
import pandas as pd
import json
#Path of a Json file
file_path="data/trends_20260411.json"
#Load Json data from file
with open(file_path,'r') as file:
    data=json.load(file)
#Convert Json file to pandas
df=pd.DataFrame(data)
#printing number of rows loaded in that data
print(f"Loaded {len(df)} stories from data/trends_20260408.json")
#Cleaning the data
# To Remove Duplicates in post_id
df=df.drop_duplicates(subset='post_id')
after =len(df)
print(f"\n After removing duplicates: {after}")
# Now removing rows with missing values in post_id,title and score
df=df.dropna(subset=['post_id','title','score'])
print(f"After removing nulls: {len(df)}")
# Changing the data types in score and num_commentes to integer type
df["score"]=df["score"].astype(int)
df["num_comments"]=df["num_comments"].astype(int)
# Removing the stories where has less than 5
df=df[df["score"]>=5]
print(f"After removing low scores: {len(df)}")
# Removing whitespaces from the Title column
df["title"]=df["title"].str.strip()
# It will create a folder of data if incase it doesn't exist
os.makedirs("data",exist_ok=True)
# Saved the clean data into CSV data
output_path="data/trends_clean.csv"
df.to_csv(output_path,index=False)
print(f"\n saved {len(df)} row to {output_path}")
#Now printng Category-wise Summary
print("\n Stories per category:")
print(df["category"].value_counts())

Loaded 110 stories from data/trends_20260408.json

 After removing duplicates: 97
After removing nulls: 97
After removing low scores: 90

 saved 90 row to data/trends_clean.csv

 Stories per category:
category
technology       23
worldnews        23
entertainment    18
sports           14
science          12
Name: count, dtype: int64


In [ ]:
#Load and Explore with pandas
import pandas as pd
#loding the file and shape of the file
df=pd.read_csv("data/trends_clean.csv")
print(f"Loded data {df.shape}")
print(f"The first 5 Rows of the dataset:")
print(df.head())
rows,columns=df.shape
print(f"\nshape of the DataFrame:{rows} rows and {columns} columns")
#Average of score and num_comment
avg_score=df["score"].mean()
print(f"\nAverage Score of the Stories: {avg_score:.2f}")
avg_num_comments=df["num_comments"].mean()
print(f"Average Number of Comments: {avg_num_comments:.2f}")

Loded data (90, 7)
The first 5 Rows of the dataset:
    post_id                                              title    category  \
0  47725403  Italo Calvino: A Traveller in a World of Uncer...  technology   
1  47721953  AI assistance when contributing to the Linux k...  technology   
2  47724921  Sam Altman's response to Molotov cocktail inci...  technology   
3  47720418  Launch HN: Twill.ai (YC S25) – Delegate to clo...  technology   
4  47669923  Clojure on Fennel Part One: Persistent Data St...  technology   

   score  num_comments        author         collected_at  
0     13             2     lermontov  2026-04-11 01:08:25  
1    152           123    hmokiguess  2026-04-11 01:08:25  
2    139           240  jack_hanford  2026-04-11 01:08:25  
3     47            46     danoandco  2026-04-11 01:08:26  
4    126            10      roxolotl  2026-04-11 01:08:26  

shape of the DataFrame:90 rows and 7 columns

Average Score of the Stories: 142.60
Average Number of Comments: 84.88
